# Stage 5 · Diagnose the dense Lego training fit

Attach the original Stage 0 kit, `lego-dense-vanilla-s42-stage3-20000.zip`, and `lego-dense-vanilla-s42-step50000.zip` as private inputs. Expanded datasets also work. Select **GPU T4 ×2** and run all cells. The dense run missed the proposal's reference window by 1.43 dB. This stage uses the two saved dense checkpoints to compare the same eight full training images at 400 × 400. It makes zero optimizer updates and uses one T4. The Stage 4 timing implies about 5 minutes 20 seconds for the 16 renders, plus setup and export.

Return **lego-dense-vanilla-s42-train-fit.zip**. The training-camera measurements help distinguish a continuing improvement in fit from errors concentrated at object boundaries or near the depth limits. They do not establish geometric ground truth.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import base64
import hashlib
import json
import os
import shutil
import statistics
import sys
import time
import zipfile

import numpy as np
from PIL import Image
import torch
from IPython.display import HTML, display

inputs = Path("/kaggle/input")
outputs = Path("/kaggle/working")
work = outputs / "recovery-stage5-dense-fit"
work.mkdir(parents=True, exist_ok=True)
if "session" not in globals():
    session = TemporaryDirectory(prefix="dense-fit-")
temp = Path(session.name)
run_id = "08ff8d2fceda43759554aa6470054853"
data_hash = "bdc22418467720a3dbf5555318532caf17c250a05433f8553ae58372f11e89e2"
code_hash = "4b85a7f9d5836739fb562c5fb3ff25fa68b063073bbea539e38ed1130f088eb5"
hashes = {
    20_000: "ab99ca798567ff5ac858982fe91500f172895a3272b65d668eab675998b30325",
    50_000: "a77230d4e17b3769445cf05e26b2bda7f094bf078418ff7d6366d906e19da8be",
}
filenames = {20_000: "lego-dense-vanilla-s42-stage3-20000.zip",
             50_000: "lego-dense-vanilla-s42-step50000.zip"}
camera_ids = [26, 86, 2, 55, 75, 93, 16, 73]
run_path = Path("runs/lego/dense-vanilla/seed-42")
chunk = 1024
device = torch.device("cuda:0")
if not torch.cuda.is_available():
    raise RuntimeError("Select the Kaggle GPU accelerator and restart this notebook.")

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def read(path):
    return json.loads(path.read_text(encoding="utf-8"))

def unpack(path, members=None):
    folder = temp / path.stem
    folder.mkdir(exist_ok=True)
    with zipfile.ZipFile(path) as archive:
        for name in (archive.namelist() if members is None else members):
            if not (folder / name).resolve().is_relative_to(folder.resolve()):
                raise ValueError(f"Unexpected member {name}")
            archive.extract(name, folder)
    return folder

expanded = sorted(inputs.rglob("RECOVERY_STAGE0.txt"))
if expanded:
    repo = expanded[0].parent
else:
    kits = sorted(inputs.rglob("stage0-input.zip"))
    if not kits:
        raise FileNotFoundError("Attach the original Stage 0 dataset.")
    repo = unpack(kits[0])

def get_snapshot(step):
    for marker in sorted(inputs.rglob("stage3-review.json")):
        record = read(marker)
        if (record.get("run_id"), record.get("checkpoint_step"), record.get("checkpoint_sha256")) == (
            run_id, step, hashes[step]
        ):
            return marker.parent
    files = sorted(inputs.rglob(filenames[step]))
    if not files:
        raise FileNotFoundError(f"Attach {filenames[step]} or its expanded dataset.")
    members = ["stage3-review.json", "manifests/lego.json",
               (run_path / "run.json").as_posix(), (run_path / "checkpoint.pt").as_posix()]
    return unpack(files[0], members)

snapshots = {step: get_snapshot(step) for step in hashes}
print(f"Selected {torch.cuda.get_device_name(device)}.")

In [ ]:
sys.path.insert(0, str(repo))
from recovery.data import verify_manifest
from recovery.health import camera_check
from recovery.media import psnr, render_maps, rgb_uint8, write_json
from recovery.training import implementation_hash, load_state, make_model, settings
from sparse_nerf.data import SceneDataset

if implementation_hash() != code_hash:
    raise ValueError("Training code differs from the reviewed run.")
media_code = (repo / "recovery/media.py").read_text(encoding="utf-8").encode()
assert hashlib.sha256(media_code).hexdigest() == "fd13945accd1ec6938b7ebceccec31fe1f8716dc2a2ad51d84068d560e09a3af"
manifest = work / "lego.json"
meta_by_step = {}
for step, root in snapshots.items():
    path = root / run_path
    meta = read(path / "run.json")
    record = read(root / "stage3-review.json")
    assert (meta["run_id"], meta["scene"], meta["protocol"], meta["seed"], meta["train_views"]) == (
        run_id, "lego", "dense-vanilla", 42, 100)
    assert record["run_id"] == run_id and record["checkpoint_step"] == step
    assert record["checkpoint_sha256"] == sha(path / "checkpoint.pt") == hashes[step]
    frozen = verify_manifest(root / "manifests/lego.json", repo / "data/blender/lego")
    assert frozen["data_hash"] == meta["data_hash"] == data_hash
    assert meta["training_config"] == settings("main", frozen["camera"])
    assert frozen["splits"]["sparse"] == camera_ids and set(camera_ids) <= set(frozen["splits"]["dense"])
    assert set(camera_ids).isdisjoint(frozen["splits"]["test"])
    meta_by_step[step] = meta
assert meta_by_step[20_000] == meta_by_step[50_000]
write_json(manifest, frozen)
# Only the eight training cameras are opened for rendering. Both checkpoints saw all 100.
dataset = SceneDataset(manifest, "sparse", meta["downscale"])
assert [dataset.frame_id(i) for i in range(len(dataset))] == [f"train/r_{k}.png" for k in camera_ids]
assert (dataset.width, dataset.height, dataset.near, dataset.far, dataset.white_background) == (400, 400, 2., 6., True)
geometry = camera_check(dataset)
print("Verified the 20,000 and 50,000 checkpoints and eight training frames.", geometry)

In [ ]:
def score_view(model, index, step):
    origins, directions, target = dataset.frame(index)
    rgb, depth, opacity = render_maps(model, dataset, (origins, directions), False, device, chunk)
    with Image.open(dataset.source / dataset.frame_id(index)) as source:
        alpha = torch.from_numpy(np.array(source.getchannel("A").resize(
            (dataset.width, dataset.height), Image.Resampling.LANCZOS)))
    foreground, background = alpha >= 128, alpha == 0
    mask = foreground.float()[None, None]
    # The silhouette band contains pixels whose 3x3 neighborhood crosses the alpha threshold.
    pool = torch.nn.functional.max_pool2d
    band = (pool(mask, 3, 1, 1) > -pool(-mask, 3, 1, 1))[0, 0]
    squared_error = (rgb.double() - target.double()).square().sum(dim=-1)
    # Expected camera-axis depth includes the far-plane residual. Exclude transparent rays.
    opaque = (alpha >= 245) & (opacity >= 0.95)
    surface = depth[opaque]
    margin = 0.05 * (dataset.far - dataset.near)
    record = dict(frame=dataset.frame_id(index), psnr=psnr(rgb, target),
                  foreground_psnr=psnr(rgb[foreground], target[foreground]),
                  white_psnr=psnr(torch.ones_like(target), target),
                  foreground_opacity=opacity[foreground].mean().item(),
                  clear_background_opacity=opacity[background].mean().item(),
                  silhouette_pixel_fraction=band.float().mean().item(),
                  silhouette_squared_error_fraction=(squared_error[band].sum() / squared_error.sum().clamp_min(1e-20)).item(),
                  opaque_depth_pixels=int(opaque.sum()), opaque_depth=None)
    if surface.numel():
        q = torch.quantile(surface, torch.tensor([0.05, 0.5, 0.95])).tolist()
        record["opaque_depth"] = dict(p05=q[0], median=q[1], p95=q[2],
            fraction_near=float((surface <= dataset.near + margin).float().mean()),
            fraction_far=float((surface >= dataset.far - margin).float().mean()))
    # The first, middle and last frozen cameras give paired previews.
    if index in (0, len(dataset) // 2, len(dataset) - 1):
        prefix = f"train-{camera_ids[index]:03d}"
        images = {f"{prefix}-{step}-rgb.png": rgb_uint8(rgb),
                  f"{prefix}-{step}-opacity.png": rgb_uint8(opacity),
                  f"{prefix}-{step}-depth.png": (
                      (depth - dataset.near) / (dataset.far - dataset.near)
                  ).clamp(0, 1).mul(65535).round().numpy().astype(np.uint16),
                  f"{prefix}-target.png": rgb_uint8(target)}
        for name, pixels in images.items():
            Image.fromarray(pixels).save(work / name)
        record["images"] = {name: sha(work / name) for name in images}
    return record

report_file = work / "dense-train-fit.json"
identity = dict(version=2, run_id=run_id, checkpoint_sha256={str(s): h for s, h in hashes.items()},
                data_hash=data_hash, code_hash=code_hash, train_camera_ids=camera_ids,
                width=400, height=400, chunk=chunk)
def evaluate_pair():
    records = {}
    means = {}
    for step, root in snapshots.items():
        state = load_state(root / run_path, meta_by_step[step])
        assert state["step"] == step
        model = make_model(meta_by_step[step]["training_config"], device)
        model.load_state_dict(state["model"])
        model.eval()
        del state
        rows = []
        for index in range(len(dataset)):
            started = time.monotonic()
            row = score_view(model, index, step)
            row["seconds"] = time.monotonic() - started
            rows.append(row)
            print(f"{step:,}  [{index+1}/8] {row['frame']}  {row['psnr']:.2f} dB  {row['seconds']:.1f} s", flush=True)
        del model
        records[str(step)] = rows
        means[str(step)] = {key: statistics.mean(r[key] for r in rows)
                            for key in ("psnr", "foreground_psnr", "white_psnr", "foreground_opacity", "clear_background_opacity",
                                        "silhouette_pixel_fraction", "silhouette_squared_error_fraction")}
    pairs = []
    for a, b in zip(records["20000"], records["50000"], strict=True):
        assert a["frame"] == b["frame"]
        pairs.append(dict(frame=b["frame"], delta_psnr=b["psnr"]-a["psnr"],
                          delta_foreground_psnr=b["foreground_psnr"]-a["foreground_psnr"]))
    return dict(identity=identity, training_config=meta_by_step[50_000]["training_config"],
                geometry=geometry, hardware=str(torch.cuda.get_device_name(device)),
                torch_version=str(torch.__version__), cuda=torch.version.cuda, frames=records, means=means,
                paired_changes=pairs, near=dataset.near, far=dataset.far,
                notes="Uniform full-image PSNR averaged per view. Alpha>=128 defines foreground. Clear background has alpha=0. Depth is expected camera-axis z with residual at far. Depth summaries require alpha>=245 and predicted opacity>=0.95. Near/far bands are the outer 5% of [2,6].")

def valid_report(result):
    if result.get("identity") != identity:
        return False
    expected = [dataset.frame_id(i) for i in range(len(dataset))]
    for step in ("20000", "50000"):
        rows = result["frames"][step]
        if [r["frame"] for r in rows] != expected:
            return False
        for row in rows:
            for name, digest in row.get("images", {}).items():
                if not (work / name).is_file() or sha(work / name) != digest:
                    return False
    return True

report = read(report_file) if report_file.is_file() else {}
if not valid_report(report):
    report = evaluate_pair()
    assert valid_report(report)
    write_json(report_file, report)
print(json.dumps(report["means"], indent=2))
print("Mean paired training PSNR change", statistics.mean(r["delta_psnr"] for r in report["paired_changes"]))

In [ ]:
review_zip = outputs / "lego-dense-vanilla-s42-train-fit.zip"
names = {name for rows in report["frames"].values() for row in rows for name in row.get("images", {})}
assert len(names) == 21 and len(report["paired_changes"]) == 8
payloads = [report_file, *(work / name for name in sorted(names))]
pending = review_zip.with_suffix(".tmp")
with zipfile.ZipFile(pending, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in payloads:
        archive.write(path, arcname=path.name)
with zipfile.ZipFile(pending) as archive:
    assert archive.testzip() is None and len(archive.namelist()) == len(payloads)
os.replace(pending, review_zip)
print(f"Download {review_zip} ({review_zip.stat().st_size:,} bytes). SHA-256 {sha(review_zip)}")
encoded = base64.b64encode(review_zip.read_bytes()).decode("ascii")
display(HTML(f'<a download="{review_zip.name}" href="data:application/zip;base64,{encoded}">Download {review_zip.name}</a>'))